# Food Access Prioritization Pipeline

Version 5 

**Trace Analysts**

This notebook loads the USDA 2019 Large Retailer Access Map (LRAM), filters to a chosen county, cleans it, creates a prioritization score, and exports analysis-ready outputs.

**How to run:** Kernel → Restart & Run All

**Design goal:** the county is a single config variable, so the whole analysis can be re-run for any other county without editing the logic.

**Data note:** Data is historical — 2010 census-tract boundaries & population, 2014–18 ACS socioeconomics, and 2019 retailer locations. It is not a 2026 measure of food access.

**Use notice:** This ranking uses historical (2010–2019) public USDA data and describes neighborhoods, not people. It identifies where to look first; it is not a funding decision. Tracts with ReviewFlag = TRUE must be checked by a person before use.

**Notebook map:**

    0. Setup & configuration
    1. Load & filter
    2. Clean & type-check 
    3. Build the prioritization score 
    4. Review flag (Mitigation)
    5. Scenario sensitivity test
    6. View & export   
    7. Validation: review-flag test

# 0. Setup & configuration
Import the libraries, set the state and county, and define the six indicators that feed the score. Everything else in the notebook reads from these settings, so this is the only cell that needs to change for a new county.

In [1]:
# Import the libraries used throughout the pipeline.
import pandas as pd
import numpy as np
from pathlib import Path
import warnings

In [2]:
# CHANGE THESE TWO LINES TO RE-RUN FOR ANOTHER AREA
STATE_NAME = "Illinois"
COUNTY_NAME = "Cook County"

# Validate the area configuration.
def _require(name, value):
    # Raise a clear error if a required config value is missing or blank.
    if value is None or str(value).strip() == "":
        raise ValueError(
            f"State and county must not be empty!")
    return str(value).strip()

# Run the validation on both required fields.
STATE_NAME = _require("STATE_NAME", STATE_NAME)
COUNTY_NAME = _require("COUNTY_NAME", COUNTY_NAME)

# Path to the raw USDA Excel file and the sheet that holds the data.
DATA_PATH = Path("FoodAccessResearchAtlasData2019.xlsx")
DATA_SHEET = "Food Access Research Atlas"

# Create the folder where the exported CSV files will be saved.
OUT_DIR = Path("outputs"); OUT_DIR.mkdir(exist_ok=True)

# The six indicators used to build the priority score.
# A higher value means a higher need for all of these except MedianFamilyIncome.
INDEX_VARS = [
    "PovertyRate",       # % population below the federal poverty line
    "lapop1share",       # % of tract >1 mi from a supermarket
    "lalowi1share",      # % low-income AND >1 mi from a supermarket
    "lahunv1share",      # % housing units with no vehicle AND >1 mi
    "lasnap1share",      # % SNAP households AND >1 mi
    "MedianFamilyIncome" # inverted: lower income = higher need
]

# Variables where a LOW value means MORE need (inverted during scoring).
INVERT = ["MedianFamilyIncome"]

# USDA stores some missing values as the literal text "NULL" instead of a blank cell.
NULLS = ["NULL", "null", "NA", "N/A", ""]

# Tracts with fewer residents than this are flagged for review in Step 4.
LOW_POP = 1000

In [3]:
# Small-county guard (Week 11 boundary test).
# Min-max scaling needs enough tracts to spread scores out. With 2 tracts every indicator becomes 0 or 1, so the score stops meaning anything.
MIN_TRACTS_STOP = 3    # fewer than this: stop with an error
MIN_TRACTS_WARN = 10   # fewer than this: run, but warn that scores are coarse

# 1. Load & filter
Load the dataset, read only the data sheet, then filter to the state and county. Keeping the filter inside the pipeline so the file is reusable.

After filtering, the small-county guard checks how many tracts were kept. In the Week 11 boundary test, a 2-tract county produced a tied ranking with no warning, so the pipeline now stops below 3 tracts and warns below 10. 

Expected result for Cook County: 1,314 rows after the filter, and the tract-count check prints OK.

In [4]:
# Load the data sheet and treat USDA "NULL" text as missing.
def load_lram(path, sheet=DATA_SHEET):
    return pd.read_excel(path, sheet_name=sheet, na_values=NULLS)

# Filter the national dataset down to the configured state and county.
def filter_area(df, state, county):
    out = df[df["State"] == state].copy()
    out = out[out["County"] == county].copy()
    return out.reset_index(drop=True)

# Stop or warn when a county has too few tracts for a meaningful ranking.
def check_tract_count(n, county, state):
    if n < MIN_TRACTS_STOP:
        raise ValueError(
            f"{county}, {state} has only {n} tract(s). At least {MIN_TRACTS_STOP} are needed "
            f"for a meaningful ranking; with so few tracts, min-max scaling turns every indicator into 0 or 1.")
    if n < MIN_TRACTS_WARN:
        warnings.warn(
            f"{county}, {state} has only {n} tracts. Scores are coarse; treat the ranking as "
            f"descriptive and review every tract by hand.")
        return "WARN"
    return "OK"

# Load the national file, then filter to the configured area.
df_all = load_lram(DATA_PATH)
df = filter_area(df_all, STATE_NAME, COUNTY_NAME)

# Stop early with a clear message if the state/county combination matched no rows.
if len(df) == 0:
    raise ValueError(
        f"No rows found for COUNTY_NAME={COUNTY_NAME!r} in STATE_NAME={STATE_NAME!r}. "
        f"Check spelling and formatting (e.g. 'Cook County', not 'Cook').")

# Stop or warn if the county is too small to rank.
TRACT_CHECK = check_tract_count(len(df), COUNTY_NAME, STATE_NAME)

# Show what was loaded and kept.
print(f"National rows loaded: {len(df_all):,}")
print(f"Rows after filter: {len(df):,} ({COUNTY_NAME}, {STATE_NAME})")
print(f"Tract-count check: {TRACT_CHECK}")
print(f"Columns: {df.shape[1]}")
df.head()

National rows loaded: 72,531
Rows after filter: 1,314 (Cook County, Illinois)
Tract-count check: OK
Columns: 147


,CensusTract,State,County,Urban,Pop2010,OHU2010,GroupQuartersFlag,NUMGQTRS,PCTGQTRS,LILATracts_1And10,...,TractSeniors,TractWhite,TractBlack,TractAsian,TractNHOPI,TractAIAN,TractOMultir,TractHispanic,TractHUNV,TractSNAP
0,17031010100,Illinois,Cook County,1,4854,2302,0,218.0,4.491141,0,...,277.0,1810.0,2437.0,150.0,1.0,33.0,423.0,616.0,1162.0,433.0
1,17031010201,Illinois,Cook County,1,6450,2463,0,163.0,2.527132,0,...,314.0,2312.0,2350.0,299.0,6.0,48.0,1435.0,2049.0,751.0,851.0
2,17031010202,Illinois,Cook County,1,2818,1115,0,315.0,11.178141,0,...,309.0,1237.0,953.0,143.0,1.0,18.0,466.0,789.0,464.0,232.0
3,17031010300,Illinois,Cook County,1,6236,2826,0,791.0,12.684413,0,...,884.0,3267.0,1722.0,346.0,16.0,21.0,864.0,1169.0,993.0,470.0
4,17031010400,Illinois,Cook County,1,5042,2098,0,1349.0,26.755256,0,...,263.0,3341.0,729.0,562.0,5.0,17.0,388.0,480.0,655.0,277.0


# 2. Clean & type-check
The na_values setting already converted "NULL" → NaN on load, so the share variables come in as numeric. This cell verifies that so nothing breaks later. CensusTract is kept as an 11-digit string because it is a geographic label, not a number (we want to preserve leading zeros).

In [5]:
# Force the index variables to numeric
# Coerce turns any stray non-number into NaN.
for col in INDEX_VARS:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Keep the tract id as an 11-digit zero-padded string (drop any trailing ".0" first).
df["CensusTract"] = df["CensusTract"].astype(str).str.split(".").str[0].str.zfill(11)

# Build a small data-quality table for the index variables.
quality = pd.DataFrame({
    "dtype":     [df[c].dtype for c in INDEX_VARS],
    "missing":   [df[c].isna().sum() for c in INDEX_VARS],
    "missing_%": [round(df[c].isna().mean()*100, 1) for c in INDEX_VARS],
    "min":       [df[c].min() for c in INDEX_VARS],
    "max":       [df[c].max() for c in INDEX_VARS],})

quality.index = INDEX_VARS
print("Data-quality summary for index variables:")
quality

Data-quality summary for index variables:


,dtype,missing,missing_%,min,max
PovertyRate,float64,0,0.0,0.293341,77.099587
lapop1share,float64,972,74.0,0.000000,100.000001
lalowi1share,float64,972,74.0,0.000000,87.673914
lahunv1share,float64,972,74.0,0.000000,53.865745
lasnap1share,float64,972,74.0,0.000000,62.007170
MedianFamilyIncome,float64,9,0.7,10131.000000,250001.000000


In the share variables a number is populated "only for tracts flagged low-access at 1 mile" so if the tract is not low-access at 1 mile it gets NaN meaning it doesn't apply so there is no number.

For scoring, NaN in the share variables will be treated as 0 to say "this tract has no low-access-at-1-mile population," which is what the blank actually means.

MedianFamilyIncome and PovertyRate blanks are different: they are true unknowns. We fill them with the county median and record which rows were filled in the _imputed columns. Those columns are reused in Step 4 to flag tracts for review.

In [6]:
# Distance-share NaNs mean "not low-access at 1 mile" -> treat as 0 for scoring.
DISTANCE_SHARES = [c for c in INDEX_VARS if c.startswith("la")]
df[DISTANCE_SHARES] = df[DISTANCE_SHARES].fillna(0.0)

# MedianFamilyIncome / PovertyRate NaNs are true unknowns -> impute with the county median
# and flag which rows were filled, so the imputation stays transparent.
for col in ["MedianFamilyIncome", "PovertyRate"]:
    if col in df.columns:
        df[col + "_imputed"] = df[col].isna()
        df[col] = df[col].fillna(df[col].median())

# Confirm there are no missing values left in the index variables.
print("Remaining missing values in index vars:", int(df[INDEX_VARS].isna().sum().sum()))

Remaining missing values in index vars: 0


# 3. Build the prioritization score
**Approach:** min–max normalize each variable to 0–1, then average.

    Put every variable on the same 0–1 scale: (x − min) / (max − min).
    Invert income (low income = high need).
    Average the normalized variables into one PriorityScore (0–1), with optional weights.

The minmax function rescales a column so its lowest value becomes 0, its highest becomes 1, and everything else lands proportionally in between.

This equal-weight PriorityScore is our baseline. Step 5 compares it with two other weightings.

Because min-max scaling uses this county's own lowest and highest values, a score only means something relative to other tracts in the same county. A 0.9 in a small county is not comparable to a 0.9 in Cook County. Scores are relative to the selected county and cannot be compared across counties.

In [7]:
# Rescale a column onto a 0-to-1 scale.
def minmax(s):
    lo, hi = s.min(), s.max()
    # If every value is identical, return all zeros (avoids divide-by-zero).
    if hi > lo:
        return (s - lo) / (hi - lo)
    else:
        return pd.Series(0.0, index=s.index)

# Equal weights by default; change a value here to emphasize an indicator.
WEIGHTS = {v: 1.0 for v in INDEX_VARS}

# Normalize each variable; invert the ones where a low value means more need.
norm = pd.DataFrame(index=df.index)
for v in INDEX_VARS:
    n = minmax(df[v])
    if v in INVERT:
        n = 1 - n
    norm[v + "_score"] = n

# Convert the weights to a Series aligned with the normalized columns.
w = pd.Series(WEIGHTS)

# PriorityScore = weighted average of the normalized components (0-1).
df["PriorityScore"] = (norm.values * w.values).sum(axis=1) / w.sum()

# Rank tracts so that rank 1 = highest priority. Ties share the lower rank number.
df["PriorityRank"] = df["PriorityScore"].rank(ascending=False, method="min").astype(int)

# Attach the normalized component columns to the main dataframe for later inspection.
# (This is done once here; Step 5 reuses `norm` instead of rebuilding it.)
df = pd.concat([df, norm], axis=1)

# Show the score range and the ten highest-priority tracts.
print("PriorityScore range:", round(df.PriorityScore.min(), 3), "->", round(df.PriorityScore.max(), 3))
df[["CensusTract", "PriorityScore", "PriorityRank"] + INDEX_VARS].sort_values("PriorityRank").head(10)

PriorityScore range: 0.003 -> 0.967


,CensusTract,PriorityScore,PriorityRank,PovertyRate,lapop1share,lalowi1share,lahunv1share,lasnap1share,MedianFamilyIncome
514,17031540102,0.967044,1,64.150943,100.000001,87.673914,53.865745,62.007170,17122.0
513,17031540101,0.884952,2,52.922318,92.261515,76.848514,51.205919,56.760179,19881.0
1269,17031838800,0.716158,3,42.514970,95.961718,57.497608,28.363315,41.911970,27083.0
1189,17031829700,0.624664,4,32.315764,96.304441,53.508441,17.252142,36.962597,48194.0
1042,17031821500,0.587037,5,40.681004,77.415171,49.390325,18.655457,29.057288,47563.0
1155,17031827700,0.579798,6,29.387187,100.000000,52.906824,8.917476,28.649237,41563.0
1026,17031820603,0.492289,7,20.894772,100.000001,38.236609,3.853674,25.352113,65545.0
1148,17031827000,0.484635,8,31.325625,67.461090,41.802874,10.868663,17.230334,40662.0
448,17031430700,0.483328,9,44.977376,47.513138,30.331301,25.749294,16.513481,69436.0
1154,17031827600,0.471462,10,25.409104,68.645694,36.044000,15.679582,17.724234,51563.0


# 4. Review flag (Mitigation)
**Why:** Section 8 identified two risks in how the score could mislead a reader. - R1: Tracts with very few residents can have unstable shares, so a few households can move the score a lot. - R2: Tracts with imputed income or poverty are partly scored on the county median, not their own data.

**What it does:** ReviewFlag is TRUE when any of these is TRUE: - LowPop_flag: Pop2010 < 1000 (added by Alex; raised from 500 in Version 4) - PovertyRate_imputed (from Step 2) - MedianFamilyIncome_imputed (from Step 2)

**What it does NOT do:** it does not change PriorityScore or PriorityRank. It only marks tracts a person must check before they go on a shortlist ("prioritize, then investigate").

**Why 1,000:** a Week 11 threshold test compared cutoffs of 250, 500, 750, 1,000, 1,500, and 2,000 residents. At 1,000 the flag catches 10 small tracts (7 more than at 500), and none of them are in the Top 20. Above that it grows quickly: at 1,500 it flags 98 low-population tracts, including rank 13 (1,327 residents), and stops being a targeted check

**Expected result for Cook County:** 10 low-population tracts, 9 imputed-income tracts, 0 imputed-poverty tracts, 16 flagged in total (some tracts have more than one reason), and 1 flagged tract in the Top 20 (rank 9, flagged for imputed income).

In [8]:
# Mark tracts that need a human check before their rank is used.
def add_review_flag(d, low_pop=LOW_POP):
    d = d.copy()
    d["LowPop_flag"] = d["Pop2010"] < low_pop
    reasons = [c for c in ["LowPop_flag", "PovertyRate_imputed", "MedianFamilyIncome_imputed"] if c in d.columns]
    d["ReviewFlag"] = d[reasons].any(axis=1)
    return d

df = add_review_flag(df)

# Summarize what was flagged and how many flagged tracts reach the Top 20.
print("Low-population tracts (<", LOW_POP, "):", int(df["LowPop_flag"].sum()))
print("Imputed income tracts:", int(df["MedianFamilyIncome_imputed"].sum()))
print("Imputed poverty tracts:", int(df["PovertyRate_imputed"].sum()))
print("Total tracts flagged for review:", int(df["ReviewFlag"].sum()))
print("Flagged tracts in the Top 20:", int(df.nsmallest(20, "PriorityRank")["ReviewFlag"].sum()))
df.loc[df["ReviewFlag"], ["CensusTract", "Pop2010", "PriorityRank", "PriorityScore",
                          "LowPop_flag", "PovertyRate_imputed", "MedianFamilyIncome_imputed"]].sort_values("PriorityRank")

Low-population tracts (< 1000 ): 10
Imputed income tracts: 9
Imputed poverty tracts: 0
Total tracts flagged for review: 16
Flagged tracts in the Top 20: 1


,CensusTract,Pop2010,PriorityRank,PriorityScore,LowPop_flag,PovertyRate_imputed,MedianFamilyIncome_imputed
448,17031430700,2210,9,0.483328,False,False,True
348,17031271800,970,46,0.316463,True,False,False
394,17031340600,907,54,0.296658,True,False,False
396,17031350400,389,56,0.294274,True,False,False
352,17031280900,942,68,0.283383,True,False,False
404,17031380500,752,83,0.266136,True,False,False
1300,17031842500,2451,95,0.262372,False,False,True
408,17031381500,237,101,0.260011,True,False,False
438,17031420700,3261,139,0.244758,False,False,True
400,17031351500,255,214,0.223918,True,False,True


# 5. Scenario sensitivity test
**Why:** the baseline gives every indicator equal weight. That is a choice, not a fact, so we test whether the ranking holds up under two other reasonable weightings.

**Scenarios (weights add up to 1.0):** - Equal: all six indicators weighted equally (same ranking as the baseline PriorityScore). - Economic_Hardship: poverty and income weighted 0.30 each; the four distance shares 0.10 each. - Food_Access: poverty and income weighted 0.10 each; the four distance shares 0.20 each.

**Decision rule (set before running):** the ranking is considered robust if each alternative scenario has a Spearman rank correlation of at least 0.90 with Equal AND keeps at least 15 of the 20 Equal Top 20 tracts.

**Note:** this step reuses the normalized norm table from Step 3, so the normalized columns are not added to df a second time.

In [9]:
SCENARIOS = {"Equal": {v: 1.0 / len(INDEX_VARS) for v in INDEX_VARS},

    "Economic_Hardship": {
        "PovertyRate": 0.30,
        "MedianFamilyIncome": 0.30,
        "lapop1share": 0.10,
        "lalowi1share": 0.10,
        "lahunv1share": 0.10,
        "lasnap1share": 0.10},

    "Food_Access": {
        "PovertyRate": 0.10,
        "MedianFamilyIncome": 0.10,
        "lapop1share": 0.20,
        "lalowi1share": 0.20,
        "lahunv1share": 0.20,
        "lasnap1share": 0.20}}

# Verify that each scenario includes every indicator.
for scenario, weights in SCENARIOS.items():
    assert set(weights) == set(INDEX_VARS), (
        f"{scenario} weights do not match INDEX_VARS.")

    assert abs(sum(weights.values()) - 1.0) < 1e-9, (
        f"{scenario} weights must add up to 1.0.")

# Calculate a separate score and rank for each scenario,
# using the normalized indicators already built in Step 3.
for scenario, weights in SCENARIOS.items():

    # Align weights with the normalized indicators.
    w = pd.Series(weights, dtype=float).reindex(INDEX_VARS)

    # Calculate the weighted average priority score.
    df[f"Score_{scenario}"] = (norm[[v + "_score" for v in INDEX_VARS]].to_numpy()
                               @ w.to_numpy()) / w.sum()

    # Rank census tracts, with rank 1 representing highest priority.
    df[f"Rank_{scenario}"] = (df[f"Score_{scenario}"].rank(ascending=False, method="min").astype(int))

In [10]:
# Display the top 20 tracts under each scenario.
for scenario in SCENARIOS:
    print(f"\nTop 20 Census Tracts: {scenario.replace('_', ' ')}")

    display(df[["CensusTract", f"Score_{scenario}", f"Rank_{scenario}"]]
            .sort_values(f"Rank_{scenario}").head(20))


Top 20 Census Tracts: Equal


,CensusTract,Score_Equal,Rank_Equal
514,17031540102,0.967044,1
513,17031540101,0.884952,2
1269,17031838800,0.716158,3
1189,17031829700,0.624664,4
1042,17031821500,0.587037,5
1155,17031827700,0.579798,6
1026,17031820603,0.492289,7
1148,17031827000,0.484635,8
448,17031430700,0.483328,9
1154,17031827600,0.471462,10



Top 20 Census Tracts: Economic Hardship


,CensusTract,Score_Economic_Hardship,Rank_Economic_Hardship
514,17031540102,0.940680,1
513,17031540101,0.859885,2
1269,17031838800,0.725504,3
1189,17031829700,0.626447,4
1042,17031821500,0.626180,5
398,17031351100,0.600000,6
1155,17031827700,0.597430,7
1180,17031829000,0.594029,8
348,17031271800,0.569634,9
448,17031430700,0.556904,10



Top 20 Census Tracts: Food Access


,CensusTract,Score_Food_Access,Rank_Food_Access
514,17031540102,0.980227,1
513,17031540101,0.897486,2
1269,17031838800,0.711486,3
1189,17031829700,0.623772,4
1155,17031827700,0.570982,5
1042,17031821500,0.567466,6
1026,17031820603,0.487026,7
1193,17031830001,0.460283,8
1176,17031828702,0.456134,9
1148,17031827000,0.453887,10


Next, put the three rankings side by side and count how many scenarios place each tract in the Top 20. A tract that is Top 20 in all three is a consistently high-priority tract, regardless of weighting.

In [11]:
# Compare the rankings and scores from all three scenarios.
comparison = df[
    ["CensusTract",
     "Score_Equal",
     "Rank_Equal",
     "Score_Economic_Hardship",
     "Rank_Economic_Hardship",
     "Score_Food_Access",
     "Rank_Food_Access",
     "ReviewFlag"]].copy()

# Identify tracts appearing in the top 20 under each scenario.
comparison["Top20_Equal"] = comparison["Rank_Equal"] <= 20
comparison["Top20_Economic_Hardship"] = (comparison["Rank_Economic_Hardship"] <= 20)
comparison["Top20_Food_Access"] = (comparison["Rank_Food_Access"] <= 20)

# Count how many scenarios place each tract in the top 20.
comparison["Top20_Count"] = comparison[["Top20_Equal", "Top20_Economic_Hardship", "Top20_Food_Access"]].sum(axis=1)

# Display the tracts that appear in the top 20 most consistently.
display(comparison.sort_values(["Top20_Count", "Rank_Equal"],
                               ascending=[False, True]).head(20))

,CensusTract,Score_Equal,Rank_Equal,Score_Economic_Hardship,Rank_Economic_Hardship,Score_Food_Access,Rank_Food_Access,ReviewFlag,Top20_Equal,Top20_Economic_Hardship,Top20_Food_Access,Top20_Count
514,17031540102,0.967044,1,0.940680,1,0.980227,1,False,True,True,True,3
513,17031540101,0.884952,2,0.859885,2,0.897486,2,False,True,True,True,3
1269,17031838800,0.716158,3,0.725504,3,0.711486,3,False,True,True,True,3
1189,17031829700,0.624664,4,0.626447,4,0.623772,4,False,True,True,True,3
1042,17031821500,0.587037,5,0.626180,5,0.567466,6,False,True,True,True,3
1155,17031827700,0.579798,6,0.597430,7,0.570982,5,False,True,True,True,3
1148,17031827000,0.484635,8,0.546132,12,0.453887,10,False,True,True,True,3
448,17031430700,0.483328,9,0.556904,10,0.446540,12,True,True,True,True,3
1180,17031829000,0.442982,13,0.594029,8,0.367459,17,False,True,True,True,3
454,17031431400,0.421498,16,0.538083,13,0.363206,19,False,True,True,True,3


In [12]:
# How many tracts appear in 3, 2, 1, or 0 of the Top 20 lists?
print(comparison["Top20_Count"].value_counts().sort_index(ascending=False))

print("Unique tracts appearing in at least one top 20:",
    (comparison["Top20_Count"] > 0).sum())


# Identify census tracts appearing in all three top 20 lists.
consistent_top_20 = comparison[comparison["Top20_Count"] == 3].sort_values("Rank_Equal")

print("Consistently high-priority census tracts:")
display(consistent_top_20)

# Calculate the percentage of the original top 20
# that appears in all three scenarios.
consistency_percentage = (len(consistent_top_20) / 20) * 100

print(f"Top 20 consistency: {consistency_percentage:.1f}%")

Top20_Count
3      11
2       8
1      11
0    1284
Name: count, dtype: int64
Unique tracts appearing in at least one top 20: 30
Consistently high-priority census tracts:


,CensusTract,Score_Equal,Rank_Equal,Score_Economic_Hardship,Rank_Economic_Hardship,Score_Food_Access,Rank_Food_Access,ReviewFlag,Top20_Equal,Top20_Economic_Hardship,Top20_Food_Access,Top20_Count
514,17031540102,0.967044,1,0.940680,1,0.980227,1,False,True,True,True,3
513,17031540101,0.884952,2,0.859885,2,0.897486,2,False,True,True,True,3
1269,17031838800,0.716158,3,0.725504,3,0.711486,3,False,True,True,True,3
1189,17031829700,0.624664,4,0.626447,4,0.623772,4,False,True,True,True,3
1042,17031821500,0.587037,5,0.626180,5,0.567466,6,False,True,True,True,3
1155,17031827700,0.579798,6,0.597430,7,0.570982,5,False,True,True,True,3
1148,17031827000,0.484635,8,0.546132,12,0.453887,10,False,True,True,True,3
448,17031430700,0.483328,9,0.556904,10,0.446540,12,True,True,True,True,3
1180,17031829000,0.442982,13,0.594029,8,0.367459,17,False,True,True,True,3
454,17031431400,0.421498,16,0.538083,13,0.363206,19,False,True,True,True,3


Top 20 consistency: 55.0%


Expected result for Cook County: - Economic_Hardship: rho = 0.953, Top 20 overlap = 11 of 20 -> correlation passes, overlap fails - Food_Access: rho = 0.975, Top 20 overlap = 19 of 20 -> passes - 11 tracts are Top 20 in all three scenarios (55.0%); 30 unique tracts appear in at least one Top 20.

Interpretation: the overall ranking is stable, but the Top 20 shifts when economic hardship is weighted more heavily. The Top 20 should be reported as a weighting-sensitive shortlist, and the 11 consistent tracts are the strongest candidates.

In [13]:
# Decision rule: rho >= 0.90 AND at least 15 of the Equal Top 20 retained.
MIN_RHO, MIN_OVERLAP = 0.90, 15
equal_top20 = set(df.loc[df["Rank_Equal"] <= 20, "CensusTract"])

rows = []
for scenario in ["Economic_Hardship", "Food_Access"]:
    rho = df["Rank_Equal"].corr(df[f"Rank_{scenario}"])  # Spearman = correlation of ranks
    overlap = len(equal_top20 & set(df.loc[df[f"Rank_{scenario}"] <= 20, "CensusTract"]))
    rows.append({
        "Scenario": scenario,
        "Spearman_rho": round(rho, 3),
        "Top20_overlap": f"{overlap} / 20",
        "rho_pass": rho >= MIN_RHO,
        "overlap_pass": overlap >= MIN_OVERLAP,})

sensitivity = pd.DataFrame(rows)
sensitivity

,Scenario,Spearman_rho,Top20_overlap,rho_pass,overlap_pass
0,Economic_Hardship,0.953,11 / 20,True,False
1,Food_Access,0.975,19 / 20,True,True


# 6. View & export
Export analysis-ready files. We keep only the columns needed for visuals — identifiers, the score and rank, the raw index variables, their normalized _score versions, the USDA LILATracts_1And10 flag (used by the LILA comparison figure), the imputation flags, and the Week 10 review flags (LowPop_flag, ReviewFlag). The scenario scores are saved to a separate file so the main results file stays focused on the baseline score.

In [14]:
# Sort tracts by priority rank for export.
ranked = df.sort_values("PriorityRank")

# Define the columns to keep in the exported files (not all 147 raw fields).
ID_COLS    = ["CensusTract", "State", "County", "Pop2010"]
SCORE_COLS = ["PriorityRank", "PriorityScore"]
FLAG_COLS  = [c for c in ["LILATracts_1And10", "PovertyRate_imputed", "MedianFamilyIncome_imputed",
                          "LowPop_flag", "ReviewFlag"]
              if c in df.columns]
NORM_COLS  = [v + "_score" for v in INDEX_VARS]

OUTPUT_COLS = ID_COLS + SCORE_COLS + FLAG_COLS + INDEX_VARS + NORM_COLS

# Keep only columns that actually exist (guards against a swapped county/variable set).
OUTPUT_COLS = [c for c in OUTPUT_COLS if c in ranked.columns]

# Build the trimmed output tables.
ranked_out = ranked[OUTPUT_COLS]
top20_out  = ranked_out.head(20)

print(f"Tracts scored: {len(df):,}")
print(f"Exporting {len(OUTPUT_COLS)} columns")

# Show the top 10 with all index variables for a quick check.
print("\nTop 10 priority tracts:")
display(top20_out[["CensusTract", "PriorityRank", "PriorityScore", "ReviewFlag"] + INDEX_VARS].head(10))

# Save the files, named for the chosen county (e.g. cook_county_scored.csv).
slug = COUNTY_NAME.lower().replace(" ", "_")
ranked_out.to_csv(OUT_DIR / f"{slug}_scored.csv", index=False)
top20_out.to_csv(OUT_DIR / f"{slug}_top20.csv", index=False)
comparison.sort_values("Rank_Equal").to_csv(OUT_DIR / f"{slug}_scenarios.csv", index=False)
print(f"\nWrote: {OUT_DIR / f'{slug}_scored.csv'}, _top20.csv, and _scenarios.csv")

Tracts scored: 1,314
Exporting 23 columns

Top 10 priority tracts:


,CensusTract,PriorityRank,PriorityScore,ReviewFlag,PovertyRate,lapop1share,lalowi1share,lahunv1share,lasnap1share,MedianFamilyIncome
514,17031540102,1,0.967044,False,64.150943,100.000001,87.673914,53.865745,62.007170,17122.0
513,17031540101,2,0.884952,False,52.922318,92.261515,76.848514,51.205919,56.760179,19881.0
1269,17031838800,3,0.716158,False,42.514970,95.961718,57.497608,28.363315,41.911970,27083.0
1189,17031829700,4,0.624664,False,32.315764,96.304441,53.508441,17.252142,36.962597,48194.0
1042,17031821500,5,0.587037,False,40.681004,77.415171,49.390325,18.655457,29.057288,47563.0
1155,17031827700,6,0.579798,False,29.387187,100.000000,52.906824,8.917476,28.649237,41563.0
1026,17031820603,7,0.492289,False,20.894772,100.000001,38.236609,3.853674,25.352113,65545.0
1148,17031827000,8,0.484635,False,31.325625,67.461090,41.802874,10.868663,17.230334,40662.0
448,17031430700,9,0.483328,True,44.977376,47.513138,30.331301,25.749294,16.513481,69436.0
1154,17031827600,10,0.471462,False,25.409104,68.645694,36.044000,15.679582,17.724234,51563.0



Wrote: outputs\cook_county_scored.csv, _top20.csv, and _scenarios.csv


# 7. Validation: review-flag test
This test reads the exported file from Step 6, independently rebuilds the review flag, and checks the results. For Cook County, it compares them against fixed reference numbers from our verified Version 5 run, so any change that shifts the results is caught. For any other county, there are no reference numbers yet, so it uses the Step 4 results as the expected values and checks that the exported file reproduces them. No numbers need to be changed by hand. If any check fails, the assert at the end stops the notebook with an error.

In [15]:
data = pd.read_csv(OUT_DIR / f"{slug}_scored.csv", dtype={"CensusTract": str})

# Expected values.
# Cook County has fixed reference numbers from our verified Version 5 run.
# Any other county has no reference numbers yet, so the expected values come from Step 4,
# and this step checks that the exported file reproduces the same flags.
if STATE_NAME == "Illinois" and COUNTY_NAME == "Cook County":
    EXPECTED = {"rows": 1314, "lowpop": 10, "imputed_income": 9, "total": 16, "top20": 1}
else:
    EXPECTED = {"rows": len(df),
                "lowpop": int(df["LowPop_flag"].sum()),
                "imputed_income": int(df["MedianFamilyIncome_imputed"].sum()),
                "total": int(df["ReviewFlag"].sum()),
                "top20": int(df.nsmallest(20, "PriorityRank")["ReviewFlag"].sum())}
    print(f"Note: fixed reference values exist only for Cook County. For {COUNTY_NAME}, "
          f"Step 7 checks that the exported file matches the Step 4 results.")

def add_review_flag_check(d):
    d = d.copy()
    d["LowPop_flag"] = d["Pop2010"] < LOW_POP
    d["ReviewFlag"] = d[["LowPop_flag", "PovertyRate_imputed", "MedianFamilyIncome_imputed"]].any(axis=1)
    return d

before = data["PriorityScore"].copy()

d = add_review_flag_check(data)
checks = {
  f"T1 rows = {EXPECTED['rows']}": len(d) == EXPECTED["rows"],
  "T2 scores unchanged by flag": (d["PriorityScore"] == before).all(),
  f"T3 low-pop (<{LOW_POP}) flagged = {EXPECTED['lowpop']}": int(d["LowPop_flag"].sum()) == EXPECTED["lowpop"],
  f"T4 imputed income flagged = {EXPECTED['imputed_income']}": int(d["MedianFamilyIncome_imputed"].sum()) == EXPECTED["imputed_income"],
  f"T5 total flagged = {EXPECTED['total']}": int(d["ReviewFlag"].sum()) == EXPECTED["total"],
  f"T6 flagged in Top 20 = {EXPECTED['top20']}": int(d.nsmallest(20, "PriorityRank")["ReviewFlag"].sum()) == EXPECTED["top20"],
  "T7 every flagged tract has a reason": d.loc[d.ReviewFlag, ["LowPop_flag","PovertyRate_imputed","MedianFamilyIncome_imputed"]].any(axis=1).all(),}

for k, v in checks.items(): print(("PASS " if v else "FAIL ") + k)

# Stop the notebook if any check failed.
assert all(checks.values()), "One or more review-flag checks failed; see FAIL lines above."
print("All review-flag checks passed.")

PASS T1 rows = 1314
PASS T2 scores unchanged by flag
PASS T3 low-pop (<1000) flagged = 10
PASS T4 imputed income flagged = 9
PASS T5 total flagged = 16
PASS T6 flagged in Top 20 = 1
PASS T7 every flagged tract has a reason
All review-flag checks passed.
